# 06 - Inference & Demo (3-Model Comparison)

Interactive inference comparing **all 3 trained models** (2014, 2015, 2016):

1. Load all 3 model checkpoints
2. Single text prediction — side-by-side comparison
3. Batch prediction on example reviews
4. Visual aspect highlighting per model
5. Custom text input

---

## 1. Setup

In [ ]:
import subprocess, sys, os

def install_if_missing(package, pip_name=None):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or package])

install_if_missing('transformers')
install_if_missing('spacy')

import spacy
try:
    spacy.load('en_core_web_sm')
except OSError:
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])

print('Dependencies ready.')

In [ ]:
PROJECT_ROOT = os.path.expanduser('~/SOTA-ModernBERT-RGAT-Joint-Aspect-Sentiment-Extraction-for-Food-Tech-Reviews')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import torch
from IPython.display import HTML, display
from src.inference import AspectSentimentPredictor, load_predictor

# Use CPU for inference demo — loads 3 models without GPU memory pressure
# Single-text inference is fast enough on CPU
device = torch.device('cpu')
print(f'Device: {device}')
print('Imports successful.')

## 2. Load All 3 Trained Models

In [ ]:
import gc

# Load ALL available model checkpoints to CPU
predictors = {}  # year -> predictor
years = ['2014', '2015', '2016']

for year in years:
    ckpt = f'checkpoints/best_model_{year}.pt'
    if os.path.exists(ckpt):
        print(f'Loading {year} model from: {ckpt}')
        predictors[year] = load_predictor(year=year, device=device)
        print(f'  ✅ {year} model loaded successfully!')
    else:
        print(f'  ⚠️  No checkpoint found for {year} — skipping')

print(f'\n📊 Loaded {len(predictors)} model(s): {", ".join(predictors.keys())}')

gc.collect()

## 3. Single Text — Compare All Models

In [ ]:
text = "The pasta was absolutely delicious but the service was terribly slow."

print(f'Input: "{text}"')
print('=' * 70)

for year, pred in predictors.items():
    predictions = pred.predict(text)
    print(f'\n🔹 Model {year}:')
    if predictions:
        for p in predictions:
            emoji = {'positive': '😊', 'negative': '😞', 'neutral': '😐', 'conflict': '⚔️'}.get(p.sentiment, '❓')
            print(f'    {emoji} "{p.aspect}" → {p.sentiment} (conf: {p.confidence:.2f}, chars {p.start}:{p.end})')
    else:
        print('    No aspects detected.')
    print('-' * 70)

In [ ]:
# Visual HTML comparison — one panel per model
legend = '''
<div style="margin-top: 12px; font-size: 13px;">
  <b>Legend:</b>
  <span style="background: #27ae60; color: white; padding: 2px 8px; border-radius: 4px;">Positive</span>
  <span style="background: #e74c3c; color: white; padding: 2px 8px; border-radius: 4px;">Negative</span>
  <span style="background: #f39c12; color: white; padding: 2px 8px; border-radius: 4px;">Neutral</span>
  <span style="background: #8e44ad; color: white; padding: 2px 8px; border-radius: 4px;">Conflict</span>
</div>
'''

panels = []
for year, pred in predictors.items():
    predictions = pred.predict(text)
    highlighted = pred.get_highlighted_html(text, predictions)
    n_aspects = len(predictions)
    panels.append(
        f'<div style="flex: 1; min-width: 280px; padding: 12px; margin: 6px; '
        f'background: #f8f9fa; border-radius: 8px; border-left: 4px solid #3498db;">'
        f'<div style="font-weight: bold; color: #2c3e50; margin-bottom: 8px; font-size: 15px;">'
        f'🏷️ Model {year} ({n_aspects} aspect{"s" if n_aspects != 1 else ""})</div>'
        f'<div style="font-size: 16px; line-height: 2;">{highlighted}</div></div>'
    )

html = (
    f'<h3 style="color: #2c3e50;">🔍 Single Text Comparison</h3>'
    f'<div style="display: flex; flex-wrap: wrap; gap: 8px;">{"".join(panels)}</div>'
    f'{legend}'
)
display(HTML(html))

## 4. Batch Prediction — All Models Side-by-Side

In [ ]:
example_reviews = [
    "The spicy ramen was incredibly flavorful and the broth was rich.",
    "Terrible pizza with a soggy crust, but the drinks were excellent.",
    "Average food, nothing special about the ambiance either.",
    "The sushi here is the best I have ever had, fresh and perfectly seasoned.",
    "Long wait times and rude staff ruined an otherwise decent meal.",
    "Loved the cheesecake but the coffee was lukewarm and bitter.",
    "Great location with a beautiful patio, although prices are a bit high.",
    "The butter chicken was creamy and aromatic, paired perfectly with garlic naan.",
]

# Print text predictions for each review, grouped by model
for i, review in enumerate(example_reviews, 1):
    print(f'\n{"="*70}')
    print(f'[{i}] "{review}"')
    print(f'{"="*70}')
    
    for year, pred in predictors.items():
        predictions = pred.predict(review)
        print(f'  📌 Model {year}: ', end='')
        if predictions:
            aspects_str = ', '.join(
                f'"{p.aspect}"→{p.sentiment}({p.confidence:.2f})' for p in predictions
            )
            print(aspects_str)
        else:
            print('No aspects detected')

In [ ]:
# Visual HTML comparison for all reviews — each review shows all 3 models
all_blocks = []

for i, review in enumerate(example_reviews, 1):
    model_panels = []
    for year, pred in predictors.items():
        predictions = pred.predict(review)
        highlighted = pred.get_highlighted_html(review, predictions)
        n_asp = len(predictions)
        model_panels.append(
            f'<div style="flex: 1; min-width: 250px; padding: 10px; '
            f'background: white; border-radius: 6px; border: 1px solid #dee2e6;">'
            f'<div style="font-weight: bold; font-size: 13px; color: #3498db; margin-bottom: 6px;">'
            f'Model {year} ({n_asp} aspect{"s" if n_asp != 1 else ""})</div>'
            f'<div style="font-size: 14px; line-height: 1.8;">{highlighted}</div></div>'
        )
    
    block = (
        f'<div style="margin: 16px 0; padding: 16px; background: #f8f9fa; '
        f'border-radius: 8px; border-left: 4px solid #2c3e50;">'
        f'<div style="font-weight: bold; margin-bottom: 10px; color: #2c3e50;">'
        f'[{i}] "{review}"</div>'
        f'<div style="display: flex; flex-wrap: wrap; gap: 8px;">'
        f'{"".join(model_panels)}</div></div>'
    )
    all_blocks.append(block)

html = (
    '<h3 style="color: #2c3e50;">🍽️ Restaurant Review Analysis — 3-Model Comparison</h3>'
    + ''.join(all_blocks)
    + legend
)
display(HTML(html))

## 5. Prediction Details Table

In [ ]:
import pandas as pd

# Collect all predictions from ALL models into a comparison table
rows = []
for review in example_reviews:
    for year, pred in predictors.items():
        predictions = pred.predict(review)
        if predictions:
            for p in predictions:
                rows.append({
                    'Review': review[:45] + '...' if len(review) > 45 else review,
                    'Model': year,
                    'Aspect': p.aspect,
                    'Sentiment': p.sentiment,
                    'Confidence': f'{p.confidence:.3f}',
                    'Position': f'{p.start}:{p.end}',
                })
        else:
            rows.append({
                'Review': review[:45] + '...' if len(review) > 45 else review,
                'Model': year,
                'Aspect': '—',
                'Sentiment': '—',
                'Confidence': '—',
                'Position': '—',
            })

pred_df = pd.DataFrame(rows)

# Color-code by model
def color_model(val):
    colors = {
        '2014': 'background-color: #ebf5fb',
        '2015': 'background-color: #eafaf1',
        '2016': 'background-color: #fdf2e9',
    }
    return colors.get(val, '')

styled = (pred_df.style
    .set_caption('Prediction Details — All Models')
    .applymap(color_model, subset=['Model'])
)
display(styled)

## 6. Agreement Analysis

How often do the 3 models agree on aspects and sentiments?

In [ ]:
# Analyze where models agree/disagree
print('🔍 Model Agreement Analysis')
print('=' * 70)

for i, review in enumerate(example_reviews, 1):
    all_preds = {}
    for year, pred in predictors.items():
        predictions = pred.predict(review)
        all_preds[year] = {p.aspect: p.sentiment for p in predictions}
    
    # Find all unique aspects across models
    all_aspects = set()
    for preds in all_preds.values():
        all_aspects.update(preds.keys())
    
    print(f'\n[{i}] "{review[:60]}..."' if len(review) > 60 else f'\n[{i}] "{review}"')
    
    if not all_aspects:
        print('  No aspects found by any model.')
        continue
    
    for aspect in sorted(all_aspects):
        sentiments = []
        for year in predictors.keys():
            s = all_preds[year].get(aspect, '—')
            sentiments.append(f'{year}:{s}')
        
        # Check agreement
        found_sentiments = [all_preds[y].get(aspect) for y in predictors if aspect in all_preds[y]]
        if len(set(found_sentiments)) == 1 and len(found_sentiments) == len(predictors):
            status = '✅ AGREE'
        elif len(found_sentiments) < len(predictors):
            status = '⚠️  PARTIAL'
        else:
            status = '❌ DISAGREE'
        
        print(f'  "{aspect}": {" | ".join(sentiments)}  [{status}]')

## 7. Try Your Own Text

In [ ]:
# Type your own review here!
custom_text = "The margherita pizza had a perfectly crispy crust but the toppings were bland."

print(f'Input: "{custom_text}"')
print('=' * 70)

panels = []
for year, pred in predictors.items():
    predictions = pred.predict(custom_text)
    highlighted = pred.get_highlighted_html(custom_text, predictions)
    n_asp = len(predictions)
    
    print(f'\n🔹 Model {year}:')
    if predictions:
        for p in predictions:
            emoji = {'positive': '😊', 'negative': '😞', 'neutral': '😐', 'conflict': '⚔️'}.get(p.sentiment, '❓')
            print(f'    {emoji} "{p.aspect}" → {p.sentiment} (conf: {p.confidence:.2f})')
    else:
        print('    No aspects detected.')
    
    panels.append(
        f'<div style="flex: 1; min-width: 280px; padding: 12px; margin: 6px; '
        f'background: #f8f9fa; border-radius: 8px; border-left: 4px solid #3498db;">'
        f'<div style="font-weight: bold; color: #2c3e50; margin-bottom: 8px;">'
        f'Model {year} ({n_asp} aspect{"s" if n_asp != 1 else ""})</div>'
        f'<div style="font-size: 16px; line-height: 2;">{highlighted}</div></div>'
    )

html = (
    f'<h3 style="color: #2c3e50;">🔍 Custom Text — 3-Model Comparison</h3>'
    f'<div style="display: flex; flex-wrap: wrap; gap: 8px;">{"".join(panels)}</div>'
    f'{legend}'
)
display(HTML(html))

## 8. Export Predictions

In [ ]:
import json

# Export predictions from ALL models as JSON
export = []
for review in example_reviews:
    entry = {'text': review, 'predictions': {}}
    for year, pred in predictors.items():
        predictions = pred.predict(review)
        entry['predictions'][year] = [p.to_dict() for p in predictions]
    export.append(entry)

os.makedirs('outputs/results', exist_ok=True)
with open('outputs/results/inference_examples.json', 'w') as f:
    json.dump(export, f, indent=2)

total_aspects = sum(
    sum(len(e['predictions'][y]) for y in e['predictions'])
    for e in export
)
print(f'Predictions exported to: outputs/results/inference_examples.json')
print(f'Total reviews: {len(export)}')
print(f'Total aspects found (across all models): {total_aspects}')
print(f'Models included: {", ".join(predictors.keys())}')

---

## Summary

| Component | Status |
|-----------|--------|
| Load all 3 models (2014, 2015, 2016) | ✅ |
| Side-by-side single text comparison | ✅ |
| Batch prediction across all models | ✅ |
| Visual HTML highlighting per model | ✅ |
| Model agreement analysis | ✅ |
| Custom text multi-model inference | ✅ |
| JSON export with all model predictions | ✅ |

> **Tip:** Compare which model finds more aspects and which has higher confidence scores to determine the strongest checkpoint.